# Model Training

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
import joblib

processed_path = Path("../data/processed")
models_path = Path("../models")
models_path.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(processed_path / "X_train.csv")
X_test = pd.read_csv(processed_path / "X_test.csv")
y_train = pd.read_csv(processed_path / "y_train.csv").squeeze()
y_test = pd.read_csv(processed_path / "y_test.csv").squeeze()

assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (2016638, 78)
Testing data: (504160, 78)


In [2]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

joblib.dump(scaler, models_path / "standard_scaler.pkl")
print("Scaler saved.")

Scaler saved.


In [3]:
random_forest = RandomForestClassifier(
    n_estimators=100, random_state=42, n_jobs=-1, class_weight="balanced"
)
random_forest.fit(X_train_scaled, y_train)
joblib.dump(random_forest, models_path / "random_forest.pkl")
print("Random Forest trained and saved.")

Random Forest trained and saved.


- Train the Model with Decision Tree

In [8]:
rf_pred = random_forest.predict(X_test_scaled)

print("Predicted distribution:")
print(pd.Series(rf_pred).value_counts().sort_index())

non_benign = np.sum(rf_pred != 0)
print(f"Non-BENIGN predictions: {non_benign}/{len(rf_pred)}")

print(classification_report(y_test, rf_pred, zero_division=0))

Predicted distribution:
0     417666
1       1334
2      25614
3       2066
4      34741
5       1056
6       1075
7       1186
8          1
9          7
10     18332
11       644
12       276
13         1
14       161
Name: count, dtype: int64
Non-BENIGN predictions: 86494/504160
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    419012
           1       0.29      0.99      0.45       390
           2       1.00      1.00      1.00     25603
           3       0.99      1.00      0.99      2057
           4       0.99      1.00      1.00     34569
           5       0.99      1.00      0.99      1046
           6       0.99      0.99      0.99      1077
           7       1.00      1.00      1.00      1186
           8       1.00      0.50      0.67         2
           9       1.00      1.00      1.00         7
          10       0.99      1.00      0.99     18139
          11       1.00      1.00      1.00       644
          12   

- Train the Model with Random Forest

In [5]:
print("Training Random Forest...")

random_forest.fit(X_train, y_train)

print("Random Forest training completed.")

Training Random Forest...
Random Forest training completed.


- Train the Model with Logistic Regression

In [11]:
from sklearn.linear_model import LogisticRegression

logistic_regression = LogisticRegression(
    max_iter=1000,
    random_state=42
)

print("Training Logistic Regression...")

logistic_regression.fit(
    X_train,
    y_train
)

print("Logistic Regression training completed.")

Training Logistic Regression...
Logistic Regression training completed.


In [15]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_lr = logistic_regression.predict(X_test)

accuracy_lr = accuracy_score(y_test, y_pred_lr)

print("Logistic Regression Accuracy:")
print(f"{accuracy_lr:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_lr
    )
)

Logistic Regression Accuracy:
0.9780

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99    419012
           1       0.63      0.03      0.06       390
           2       1.00      0.98      0.99     25603
           3       0.97      0.86      0.91      2057
           4       1.00      0.94      0.97     34569
           5       0.85      0.84      0.85      1046
           6       0.97      0.89      0.93      1077
           7       0.95      0.98      0.96      1186
           8       1.00      1.00      1.00         2
           9       0.33      0.29      0.31         7
          10       0.73      0.99      0.84     18139
          11       1.00      0.90      0.94       644
          12       0.00      0.00      0.00       294
          13       0.00      0.00      0.00         4
          14       1.00      0.02      0.05       130

    accuracy                           0.98    504160
   macro avg       

In [16]:
from sklearn.tree import DecisionTreeClassifier

print("Training Decision Tree...")

decision_tree = DecisionTreeClassifier(
    random_state=42
)

decision_tree.fit(
    X_train,
    y_train
)

print("Decision Tree training completed.")

Training Decision Tree...
Decision Tree training completed.


In [17]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_dt = decision_tree.predict(X_test)

accuracy_dt = accuracy_score(
    y_test,
    y_pred_dt
)

print("=" * 70)
print("DECISION TREE EVALUATION")
print("=" * 70)

print(f"Accuracy: {accuracy_dt:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_dt
    )
)

DECISION TREE EVALUATION
Accuracy: 0.9981

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    419012
           1       0.82      0.86      0.84       390
           2       1.00      1.00      1.00     25603
           3       0.99      0.99      0.99      2057
           4       1.00      1.00      1.00     34569
           5       0.98      0.99      0.99      1046
           6       0.99      0.99      0.99      1077
           7       1.00      1.00      1.00      1186
           8       1.00      0.50      0.67         2
           9       0.75      0.86      0.80         7
          10       0.99      0.98      0.99     18139
          11       1.00      1.00      1.00       644
          12       0.73      0.73      0.73       294
          13       1.00      0.75      0.86         4
          14       0.42      0.42      0.42       130

    accuracy                           1.00    504160
   macro avg  

In [ ]:
import joblib
from pathlib import Path

models_path = Path("../models")

joblib.dump(
    decision_tree,
    models_path / "decision_tree.pkl"
)

joblib.dump(
    random_forest,
    models_path / "random_forest.pkl"
)

joblib.dump(
    logistic_regression,
    models_path / "logistic_regression.pkl"
)

print("All trained models saved successfully.")

All trained models saved successfully.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

print("=" * 70)
print("RANDOM FOREST VALIDATION")
print("=" * 70)

# Test on a manageable sample
TEST_SAMPLE_SIZE = 10000

X_eval = X_test.iloc[:TEST_SAMPLE_SIZE]
y_eval = y_test.iloc[:TEST_SAMPLE_SIZE]

rf_pred = random_forest.predict(X_eval)

print("\nActual distribution:")
print(pd.Series(y_eval).value_counts().sort_index())

print("\nPredicted distribution:")
print(pd.Series(rf_pred).value_counts().sort_index())

print("\nClassification Report:")
print(
    classification_report(
        y_eval,
        rf_pred,
        zero_division=0
    )
)

print("\nNon-BENIGN predictions:")

non_benign = np.sum(rf_pred != 0)

print(f"Non-BENIGN predictions: {non_benign}/{TEST_SAMPLE_SIZE}")
print(f"Percentage: {(non_benign / TEST_SAMPLE_SIZE) * 100:.2f}%")

RANDOM FOREST VALIDATION

Actual distribution:
Label
0     8296
1        9
2      519
3       35
4      693
5       19
6       20
7       19
10     368
11      14
12       7
14       1
Name: count, dtype: int64

Predicted distribution:
0     8269
1       27
2      519
3       35
4      697
5       19
6       20
7       19
10     372
11      14
12       5
14       4
Name: count, dtype: int64

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      8296
           1       0.33      1.00      0.50         9
           2       1.00      1.00      1.00       519
           3       1.00      1.00      1.00        35
           4       0.99      1.00      1.00       693
           5       1.00      1.00      1.00        19
           6       1.00      1.00      1.00        20
           7       1.00      1.00      1.00        19
          10       0.99      1.00      0.99       368
          11       1.00      1.00      1.

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

print("=" * 70)
print("RANDOM FOREST VALIDATION")
print("=" * 70)

TEST_SAMPLE_SIZE = 10000

X_eval = X_test.iloc[:TEST_SAMPLE_SIZE]
y_eval = y_test.iloc[:TEST_SAMPLE_SIZE]

rf_pred = random_forest.predict(X_eval)

print("\nActual distribution:")
print(pd.Series(y_eval).value_counts().sort_index())

print("\nPredicted distribution:")
print(pd.Series(rf_pred).value_counts().sort_index())

print("\nClassification Report:")
print(
    classification_report(
        y_eval,
        rf_pred,
        zero_division=0
    )
)

non_benign = np.sum(rf_pred != 0)

print("\nNon-BENIGN predictions:")
print(f"{non_benign}/{TEST_SAMPLE_SIZE}")
print(f"Percentage: {(non_benign / TEST_SAMPLE_SIZE) * 100:.2f}%")

RANDOM FOREST VALIDATION

Actual distribution:
Label
0     8296
1        9
2      519
3       35
4      693
5       19
6       20
7       19
10     368
11      14
12       7
14       1
Name: count, dtype: int64

Predicted distribution:
0     8269
1       27
2      519
3       35
4      697
5       19
6       20
7       19
10     372
11      14
12       5
14       4
Name: count, dtype: int64

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      8296
           1       0.33      1.00      0.50         9
           2       1.00      1.00      1.00       519
           3       1.00      1.00      1.00        35
           4       0.99      1.00      1.00       693
           5       1.00      1.00      1.00        19
           6       1.00      1.00      1.00        20
           7       1.00      1.00      1.00        19
          10       0.99      1.00      0.99       368
          11       1.00      1.00      1.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("=" * 70)
print("RANDOM FOREST FINAL EVALUATION")
print("=" * 70)

print(f"\nAccuracy:  {accuracy_score(y_eval, rf_pred):.4f}")
print(
    f"Precision: {precision_score(y_eval, rf_pred, average='weighted', zero_division=0):.4f}"
)
print(
    f"Recall:    {recall_score(y_eval, rf_pred, average='weighted', zero_division=0):.4f}"
)
print(
    f"F1-score:  {f1_score(y_eval, rf_pred, average='weighted', zero_division=0):.4f}"
)

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_eval,
        rf_pred,
        zero_division=0
    )
)

RANDOM FOREST FINAL EVALUATION

Accuracy:  0.9971
Precision: 0.9985
Recall:    0.9971
F1-score:  0.9976

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      8296
           1       0.33      1.00      0.50         9
           2       1.00      1.00      1.00       519
           3       1.00      1.00      1.00        35
           4       0.99      1.00      1.00       693
           5       1.00      1.00      1.00        19
           6       1.00      1.00      1.00        20
           7       1.00      1.00      1.00        19
          10       0.99      1.00      0.99       368
          11       1.00      1.00      1.00        14
          12       1.00      0.71      0.83         7
          14       0.25      1.00      0.40         1

    accuracy                           1.00     10000
   macro avg       0.88      0.98      0.89     10000
weighted avg       1.00      1.00      1.00     10000



In [ ]:
import joblib
from pathlib import Path

models_path = Path("../models")

joblib.dump(
    random_forest,
    models_path / "random_forest.pkl"
)

print("New balanced Random Forest saved successfully.")
print(models_path / "random_forest.pkl")

New balanced Random Forest saved successfully.
..\models\random_forest.pkl
